# Query and Generate Query Answers using a Bedrock Knowledge Base
- May 2026
- Numantic Solutions (numanticsolutions.com)

In [22]:
import os, sys
import time
import pandas as pd

# AWS
import boto3

# Object for retrieving and generating AI results from a Bedrock Knowledge Base
import bedrock_kb_query as bkbq

# Tools for loading data to S3
import s3_data_load as sdl

# Numantic utilities
utils_path = "../utils"
sys.path.insert(0, utils_path)
from utils import ApiAuthentication
api_configs = ApiAuthentication(client="Numantic")



## Read test data

In [6]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
multi_pas_qs = "multi_passage_answer_questions.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))


## Test the single-passage questions to establish baseline RAG performance

In [7]:
# Set up some parameters
aws_profile = "ns-admin"
aws_region = 'us-east-2'
kb_id = "IKMZTZONOU"
s3_bucket = "rag-search-tests"
s3_path = "documents/"


In [12]:
# Run through each single passage question
result_rows = []
for idx in df_spqs.index:

    # Create a query object
    test_querier = bkbq.BedrockKBRetriever(aws_profile=aws_profile,
                                           aws_region=aws_region,
                                           kb_id=kb_id,
                                           s3_bucket=s3_bucket,
                                           s3_path=s3_path)

    # Create a query and
    query = df_spqs.loc[idx, "question"]
    test_querier.retrieve_and_generate_results(query=query)

    # Check if correct source found
    if len(test_querier.df_rag_res.loc[0, "rag_citations"]) > 0 and \
            "doc_{}".format(df_spqs.loc[idx, "document_index"]) in \
                test_querier.df_rag_res.loc[0, "rag_citations"]:
        found_correct_source = True
    else:
        found_correct_source = False

    sum_dict = dict(query=query,
                    rag_answer=test_querier.df_rag_res.loc[0, "rag_answer"],
                    answer=df_spqs.loc[idx, "answer"],
                    rag_citations=test_querier.df_rag_res.loc[0, "rag_citations"],
                    source_document_index="doc_{}.txt".format(df_spqs.loc[idx, "document_index"]),
                    found_correct_source=found_correct_source
                    )

    result_rows.append(sum_dict)

    time.sleep(5)

# Put results into a dataframe
df_results = pd.DataFrame(data=result_rows)


## Examine test results

In [13]:
df_results

,query,rag_answer,answer,rag_citations,source_document_index,found_correct_source
0,What do keybullet kin drop?,Keybullet Kin drop a key upon death. Jammed Ke...,Keybullet kin drop a key upon death.,[doc_0],doc_0.txt,True
1,What kind of gun does the bandana bullet kin use?,The Bandana Bullet Kin wields a Machine Pistol.,The bandana bullet kin wields a machine pistol.,[doc_0],doc_0.txt,True
2,What do the giants look like?,"There are two giants described. One is burly, ...","One giant is burly, grey-skinned, and 20 feet ...",[doc_1],doc_1.txt,True
3,What happens on day 2?,I could not find specific information about wh...,"After a few miles of winding tunnel, you emerg...",[],doc_1.txt,False
4,What were the requirements for the project?,The project requirements included having a cha...,The tool had the following requirements:\n- Ch...,[doc_2],doc_2.txt,True
5,What data did was used to test the prototype?,The prototype was tested using Wikipedia pages...,Grace Hopper's Wikipedia page and Alan Turing'...,[doc_2],doc_2.txt,True
6,How do the data storage options compare?,The data storage options vary in terms of setu...,For fast start: use SQLite3 and ChromaDB (File...,[doc_3],doc_3.txt,True
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was adde...,UTF-8 support was added for European languages...,[doc_3],doc_3.txt,True
8,How do I make a button?,"To make a button in marimo, import the marimo ...",import marimo as mo\n\nbutton = mo.ui.run_butt...,[doc_4],doc_4.txt,True
9,When might I use caching?,You might use caching when you have expensive ...,"You might use caching when, for example, your ...",[doc_4],doc_4.txt,True


## Manually review RAG responses and identify questions RAG wasn't able to answer

In [15]:
no_ans_idx = [3, 10, 23, 35, 36]

df_results.loc[no_ans_idx]


,query,rag_answer,answer,rag_citations,source_document_index,found_correct_source
3,What happens on day 2?,I could not find specific information about wh...,"After a few miles of winding tunnel, you emerg...",[],doc_1.txt,False
10,What are the key topics of this article?,"I cannot determine the key topics of ""this art...","The key topics of this article are: ""why prior...","[doc_19, doc_8]",doc_5.txt,False
23,For what work did I receive criticism for my r...,"Sorry, I am unable to assist you with this req...",You received criticism for your research on sp...,[],doc_11.txt,False
35,When is this game set?,"Sorry, I am unable to assist you with this req...","This game is set in 2023, thirteen years after...",[],doc_17.txt,False
36,Who wrote 'Divine Rivals'?,I could not find information about who wrote '...,Rebecca Ross wrote 'Divine Rivals'.,[],doc_18.txt,False


## Add a Filter to the Bedrock Knowledge Base search for problematic questions

In [16]:
# Run through each single passage question
result_rows = []
for idx in no_ans_idx:

    # Create a query object
    test_querier = bkbq.BedrockKBRetriever(aws_profile=aws_profile,
                                           aws_region=aws_region,
                                           kb_id=kb_id,
                                           s3_bucket=s3_bucket,
                                           s3_path=s3_path)

    # Create a query
    query = df_spqs.loc[idx, "question"]

    # Create metadata filter
    metadata_filters = {"document_index": "doc_{}".format(df_spqs.loc[idx, "document_index"])}

    test_querier.retrieve_and_generate_results(query=query,
                                               metadata_filters=metadata_filters)

    # Check if correct source found
    if len(test_querier.df_rag_res.loc[0, "rag_citations"]) > 0 and \
            "doc_{}".format(df_spqs.loc[idx, "document_index"]) in \
                test_querier.df_rag_res.loc[0, "rag_citations"]:
        found_correct_source = True
    else:
        found_correct_source = False

    sum_dict = dict(query=query,
                    rag_answer=test_querier.df_rag_res.loc[0, "rag_answer"],
                    answer=df_spqs.loc[idx, "answer"],
                    rag_citations=test_querier.df_rag_res.loc[0, "rag_citations"],
                    source_document_index="doc_{}.txt".format(df_spqs.loc[idx, "document_index"]),
                    found_correct_source=found_correct_source
                    )

    result_rows.append(sum_dict)

    time.sleep(5)

# Put results into a dataframe
df_results_f = pd.DataFrame(data=result_rows)


In [17]:
df_results_f

,query,rag_answer,answer,rag_citations,source_document_index,found_correct_source
0,What happens on day 2?,"On day 2, after traveling a few miles through ...","After a few miles of winding tunnel, you emerg...",[doc_1],doc_1.txt,True
1,What are the key topics of this article?,The article focuses on how to maximize impact ...,"The key topics of this article are: ""why prior...",[doc_5],doc_5.txt,True
2,For what work did I receive criticism for my r...,"Sorry, I am unable to assist you with this req...",You received criticism for your research on sp...,[],doc_11.txt,False
3,When is this game set?,"Alan Wake 2 is set in 2023, thirteen years aft...","This game is set in 2023, thirteen years after...",[doc_17],doc_17.txt,True
4,Who wrote 'Divine Rivals'?,I could not find any information about who wro...,Rebecca Ross wrote 'Divine Rivals'.,[],doc_18.txt,False


## Feed source document to AI client for problematic questions

In [23]:
### Step 1. Set up session
region_name = "us-east-2"
profile_name = "ns-admin"
session = boto3.Session(profile_name=profile_name,
                        region_name=region_name)

### Step 2. Export documents to S3 in Bedrock-friendly format
s3_client = session.client('s3')
bucket = 'rag-search-tests'
s3_prefix = 'documents/'


In [30]:
# Run through each single passage question
result_rows = []
for idx in no_ans_idx:

    # Create a query object
    test_querier = bkbq.BedrockKBRetriever(aws_profile=aws_profile,
                                           aws_region=aws_region,
                                           kb_id=kb_id,
                                           s3_bucket=s3_bucket,
                                           s3_path=s3_path)

    # Create a query
    query = df_spqs.loc[idx, "question"]

    # Read source text
    file_key = "{}doc_{}.txt".format(s3_prefix,
                                     df_spqs.loc[idx, "document_index"])
    doc_txt = sdl.read_text_file_from_s3(s3_client=s3_client,
                                         bucket=bucket,
                                         file_key=file_key)
    # Query AI client
    test_querier.query_ai_direct(query=query,
                                 source_text=doc_txt)

    # All sources are correct since we're feeding the model source text
    found_correct_source = True

    sum_dict = dict(query=query,
                    rag_answer=test_querier.client_response,
                    answer=df_spqs.loc[idx, "answer"],
                    rag_citations=["doc_{}".format(df_spqs.loc[idx, "document_index"])],
                    source_document_index="doc_{}.txt".format(df_spqs.loc[idx, "document_index"]),
                    found_correct_source=found_correct_source
                    )

    result_rows.append(sum_dict)

    time.sleep(5)

# Put results into a dataframe
df_results_c = pd.DataFrame(data=result_rows)


In [31]:
df_results_c


,query,rag_answer,answer,rag_citations,source_document_index,found_correct_source
0,What happens on day 2?,"On Day 2, the party encounters **2 Ropers** in...","After a few miles of winding tunnel, you emerg...",[doc_1],doc_1.txt,True
1,What are the key topics of this article?,The key topics of this article are:\n\n1. **Th...,"The key topics of this article are: ""why prior...",[doc_5],doc_5.txt,True
2,For what work did I receive criticism for my r...,"Based on the input text, you received criticis...",You received criticism for your research on sp...,[doc_11],doc_11.txt,True
3,When is this game set?,"Based on the input text, Alan Wake 2 is set in...","This game is set in 2023, thirteen years after...",[doc_17],doc_17.txt,True
4,Who wrote 'Divine Rivals'?,Rebecca Ross wrote 'Divine Rivals'.,Rebecca Ross wrote 'Divine Rivals'.,[doc_18],doc_18.txt,True


## Some additional features not part of our RAG tests

## Get available foundational models

In [32]:
# Set up some parameters
aws_profile = "ns-admin"
aws_region = 'us-east-2'
kb_id = "IKMZTZONOU"
s3_bucket = "rag-search-tests"
s3_path = "documents/"


In [35]:
# Set up retriever object
srch_analyzer = bkbq.BedrockKBRetriever(aws_profile=aws_profile,
                                        aws_region=aws_region,
                                        kb_id=kb_id,
                                        s3_bucket=s3_bucket,
                                        s3_path=s3_path)

# Set queries
srch_analyzer.get_bedrock_foundational_models()


In [36]:
# Display some models
srch_analyzer.bedrock_models.head()


,model_name,model_id,model_provider,input_modalities,output_modalities,infer_types,model_lifecycle
0,Nova Pro,amazon.nova-pro-v1:0,Amazon,"[TEXT, IMAGE, VIDEO]",[TEXT],[INFERENCE_PROFILE],"{'status': 'ACTIVE', 'startOfLifeTime': 2024-1..."
1,Nova Micro,amazon.nova-micro-v1:0,Amazon,[TEXT],[TEXT],[INFERENCE_PROFILE],"{'status': 'ACTIVE', 'startOfLifeTime': 2024-1..."
2,Nova Lite,amazon.nova-lite-v1:0,Amazon,"[TEXT, IMAGE, VIDEO]",[TEXT],[INFERENCE_PROFILE],"{'status': 'ACTIVE', 'startOfLifeTime': 2024-1..."
3,Titan Text Embeddings V2,amazon.titan-embed-text-v2:0,Amazon,[TEXT],[EMBEDDING],[ON_DEMAND],"{'status': 'ACTIVE', 'startOfLifeTime': 2024-0..."
4,Nova Premier,amazon.nova-premier-v1:0:8k,Amazon,"[TEXT, IMAGE, VIDEO]",[TEXT],[],"{'status': 'LEGACY', 'startOfLifeTime': 2025-0..."


## Retrieve only search results

In [39]:
# Set up retriever object
srch_analyzer = bkbq.BedrockKBRetriever(aws_profile=aws_profile,
                                        aws_region=aws_region,
                                        kb_id=kb_id,
                                        s3_bucket=s3_bucket,
                                        s3_path=s3_path)

# Set queries
queries = df_spqs["question"].tolist()

# Search documents for text related to queries without filtering
srch_analyzer.retrieve_query_results(query=queries[0])


In [41]:
srch_analyzer.df_ret_res

,query,search_res,score,s3_loc,source_type,doc_metadata
0,What do keybullet kin drop?,"However, if the player does not manage to kill...",0.684259,s3://rag-search-tests/documents/doc_0.txt,gaming,{'metadataAttributes': {'document_index': 'doc...
1,What do keybullet kin drop?,Chance Kin Chance Kin run away from the player...,0.534917,s3://rag-search-tests/documents/doc_0.txt,gaming,{'metadataAttributes': {'document_index': 'doc...
2,What do keybullet kin drop?,Bullet Kin Bullet Kin are one of the most comm...,0.529086,s3://rag-search-tests/documents/doc_0.txt,gaming,{'metadataAttributes': {'document_index': 'doc...
3,What do keybullet kin drop?,Incapacitated Bullet Kin can be found in the O...,0.512837,s3://rag-search-tests/documents/doc_0.txt,gaming,{'metadataAttributes': {'document_index': 'doc...
4,What do keybullet kin drop?,"Trivia Its subtitle references Old Faithful, a...",0.510482,s3://rag-search-tests/documents/doc_0.txt,gaming,{'metadataAttributes': {'document_index': 'doc...
